In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("init_load_flag", "0", "Initial Load Flag")

In [0]:
init_load_flag = int(dbutils.widgets.get('init_load_flag'))

### **Data Reading**

In [0]:
df_customers = spark.sql("select * from databricks_ete_project.silver.silver_customers")

In [0]:
df_customers.display()

### **Removing Duplicates**

In [0]:
df_customers = df_customers.dropDuplicates(subset = ['customer_id'])

### **Removing _rescued_data column**

In [0]:
df_customers = df_customers.drop('_rescued_data')

## **Dividing New vs Old Records**

In [0]:
if init_load_flag == 0:
    df_customers_old = spark.sql("""select dim_customer_key, customer_id, create_date, update_date 
                                    from databricks_ete_project.gold.gold_customers
                                    """)

else:
    df_customers_old = spark.sql("""select 0 dim_customer_key, 0 customer_id, 0 create_date, 0 update_date 
                                    from databricks_ete_project.silver.silver_customers
                                    where 1 = 0
                                    """)

In [0]:
df_customers_old.display()

### **Renaming Columns of df_customers_old**

In [0]:
df_customers_old = df_customers_old.withColumnRenamed('customer_id', 'old_customer_id')\
                                   .withColumnRenamed('create_date', 'old_create_date')\
                                   .withColumnRenamed('update_date', 'old_update_date')\
                                   .withColumnRenamed('dim_customer_key', 'old_dim_customer_key')

### **Applying Joins With The Old Records**

In [0]:
df_join = df_customers.join(df_customers_old, df_customers.customer_id == df_customers_old.old_customer_id, 'left')
df_join.display()

### **Separating New vs Old Records**

In [0]:
df_new = df_join.filter(df_join['old_dim_customer_key'].isNull())
df_new.display()

In [0]:
df_old= df_join.filter(df_join['old_dim_customer_key'].isNotNull())
df_old.display()

### **Preparing df_old**

In [0]:
# dropping all the columns which are not required

df_old = df_old.drop('old_customer_id', 'old_update_date')

# Renaming 'old_create_date' column to 'create_date' and 'old_dim_customer_key' to 'dim_customer_key'

df_old = df_old.withColumnRenamed('old_create_date', 'create_date')\
               .withColumn('create_date', col('create_date').cast('timestamp'))\
               .withColumnRenamed('old_dim_customer_key', 'dim_customer_key')

# Recreating update_date column with the current_timestamp()

df_old = df_old.withColumn('update_date', current_timestamp())


In [0]:
df_old.display()

### **Preparing df_new**

In [0]:
df_new.display()

In [0]:
# dropping all the columns which are not required

df_new = df_new.drop('old_customer_id', 'old_dim_customer_key', 'old_update_date', 'old_create_date')

# Recreating 'update_date' and 'create_date' columns with the current_timestamp()

df_new = df_new.withColumn('update_date', current_timestamp())\
               .withColumn('create_date', current_timestamp())


In [0]:
df_new.display()

### **Surrogate Key - From 1**

In [0]:
df_new = df_new.withColumn('dim_customer_key', monotonically_increasing_id()+1)

In [0]:
df_new.display()

### **Adding Max Surrogate Key**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(dim_customer_key) as max_surrogate_key from databricks_ete_project.gold.gold_customers")
    # Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']    
    print(df_maxsur.collect())

In [0]:
df_new = df_new.withColumn('dim_customer_key', lit(max_surrogate_key)+col('dim_customer_key'))
df_new.display()

### **Union of df_old and df_new**

In [0]:
df_final = df_new.unionByName(df_old)
df_final.display()

### **SCD Type - 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_ete_project.gold.gold_customers"):
    dlt_obj = DeltaTable.forName(spark, "databricks_ete_project.gold.gold_customers")
    dlt_obj.alias("trg").merge(
        df_final.alias("src"),
        "trg.dim_customer_key = src.dim_customer_key"
    ).whenMatchedUpdateAll()\
     .whenNotMatchedInsertAll()\
     .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable("databricks_ete_project.gold.gold_customers")

In [0]:
%sql
select * from databricks_ete_project.gold.gold_customers